In [0]:
# Imports and Variable Set Up
import os
import logging
import uuid
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, PyPDFLoader
# import pyspark.pandas as ps

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)
catalog = "workspace"
schema = "ai_project"
volume = "raw_data"
vol_path = f"/Volumes/{catalog}/{schema}/{volume}/"

In [0]:
valid_extensions = ('.txt', '.pdf')
to_process = []

# Check for files
try:
    raw_files = dbutils.fs.ls(vol_path)
    if not raw_files:
        logger.warning(f"⚠️ Volume is empty: {vol_path}")
    else:
        logger.info(f"✅ Found {len(raw_files)} files to process.")

        # Validate file size
        for file in raw_files:
            if file.name.lower().endswith(valid_extensions):
                size_kb = file.size / 1024
                if size_kb < 1:
                    logger.error(f"❌ Skipping {file.name}: File is too small ({size_kb:.2f} KB).")
                    continue

                # Store the full path and extension for the next step
                to_process.append({
                    "path": file.path,
                    "name": file.name,
                    "type": "pdf" if file.name.lower().endswith(".pdf") else "text"
                })
                logger.info(f"📖 {file.name} validated ({size_kb:.2f} KB).")
            else:
                logger.warning(f"⚠️  {file} is not a permitted file type.")

        logger.info(f"✅ Total files ready for ingestion: {len(to_process)}")

except Exception as e:
    logger.error(f"❌ Error accessing volume: {e}")


In [0]:
# Set up silver tale and text splitter for chunking
silver_table = f"{catalog}.{schema}.processed_chunks"
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True,
    separators=["\n\n", "\n", ". ", " ", ""])

for file_info in to_process:
    try:
        logger.info(f"🚀 Processing: {file_info['name']}")

        # Read file (handling based on type if needed later)
        # with open(file_info['path'].replace("dbfs:", ""), "r", encoding="utf-8") as f:
        #     full_text = f.read()

        local_path = file_info['path'].replace("dbfs:", "")
        
        if file_info['type'] == "pdf":
            loader = PyPDFLoader(local_path)
        else:
            loader = TextLoader(local_path, encoding="utf-8")

        raw_docs = loader.load()

        # Create Chunks
        # chunks = text_splitter.split_text(full_text)
        chunks = text_splitter.split_documents(raw_docs)
        
        # Prepare data with a Unique ID (Required for Vector Search)
        # Add a 'chunk_id' so the vector index can track specific rows
        data = [{
            "chunk_id": str(uuid.uuid4()),
            "content": chunk.page_content, 
            "source": file_info['name'],
            "type": file_info['type'],
            "page_number": chunk.metadata.get("page", 1), # PDF page number, txt defaults to 1
            "start_index": chunk.metadata.get("start_index", 0) # char position for txt files
        } for chunk in chunks]

        # Convert to Spark DF 
        df = spark.createDataFrame(data)

        # Write to Delta with CDF Enabled
        # We check if table exists to decide between 'overwrite' (first run) and 'append'
        if not spark.catalog.tableExists(silver_table):
            (df.write.format("delta")
               .option("delta.enableChangeDataFeed", "true") # CRITICAL for AI Sync
               .mode("overwrite")
               .saveAsTable(silver_table))
            logger.info(f"✨ Created new table: {silver_table}")
        else:
            df.write.format("delta").mode("append").saveAsTable(silver_table)
            logger.info(f"➕ Appended {len(chunks)} chunks to {silver_table}")

    except Exception as e:
        logger.error(f"❌ Failed to process {file_info['name']}: {e}")

logger.info("🏁 All files processed and synced to Silver Layer.")


In [0]:
# Quick check of the new Silver Table
display(spark.sql(f"""
  SELECT source, page_number, start_index, content 
  FROM {silver_table} 
  LIMIT 5
"""))

# Check for any unusually small chunks
stats_df = spark.sql(f"SELECT len(content) as chunk_len FROM {silver_table}")
display(stats_df.summary())